# Chapter 8 — Trusted Local Resources and a Workspace-Aware Coding Agent

This chapter keeps resource discovery in the Coding Agent product layer and installs useful host-authority Tools explicitly around the reusable Runtime.

## Goal and Previous Limitation

Chapter 7 can accept explicit Extensions and freeze effective hashes, but it neither discovers local agent resources nor provides a safe-by-construction coding preset. We begin from the immutable Chapter 7 Checkpoint and add those product decisions without putting implicit Tools or filesystem scanning into `AgentRuntime`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
CHAPTER_7 = ROOT / 'course' / 'checkpoints' / 'ch07'
if not (CHAPTER_7 / 'checkpoint.json').is_file():
    raise RuntimeError('run this Chapter Notebook from the repository root')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Conceptual Model

`ResourceLoader` is the deep module at the local discovery and Project Trust seam. It resolves explicit, project, user, then built-in resources; Context Files remain an independent ordinary-context input. `PromptAssembler` owns deterministic ordering and exposes only Skill metadata until activation. `CodingToolPreset` is a second deep module: callers explicitly install read, write, edit, and real Bash while file paths remain confined and text mutation remains atomic.

Project Trust decides which behavior-changing startup inputs may load. The Workspace Boundary constrains only file Tools. Bash and Extensions retain the authority of the host process and are not sandbox mechanisms.

## Minimal Execution

The following Export Cells add the two Chapter 8 modules and replace the public package surface. Each cell owns one complete file.

In [ ]:
RESOURCES_SOURCE = '"""Trusted, deterministic local resources for the optional Coding Agent layer."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable, Mapping, Sequence\nfrom dataclasses import dataclass, field, replace\nfrom enum import Enum\nfrom hashlib import sha256\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom types import MappingProxyType\nfrom typing import Protocol\n\nfrom .tools import Tool\n\n\n_RESOURCE_NAME = re.compile(r"[a-z0-9][a-z0-9_-]{0,63}\\Z")\n\n\nclass ResourceScope(str, Enum):\n    EXPLICIT = "explicit"\n    PROJECT = "project"\n    USER = "user"\n    BUILTIN = "builtin"\n\n\nclass ResourceKind(str, Enum):\n    SKILL = "skill"\n    CONTEXT = "context"\n    SYSTEM = "system"\n    APPEND_SYSTEM = "append_system"\n    PROMPT_TEMPLATE = "prompt_template"\n    EXTENSION = "extension"\n    SETTINGS = "settings"\n\n\n@dataclass(frozen=True, slots=True)\nclass Resource:\n    kind: ResourceKind\n    name: str\n    description: str\n    content: str\n    source: str\n    scope: ResourceScope = ResourceScope.EXPLICIT\n    related_resources: Mapping[str, str] = field(default_factory=dict)\n\n    @classmethod\n    def skill(\n        cls,\n        name: str,\n        description: str,\n        content: str,\n        *,\n        source: str,\n        related_resources: Mapping[str, str] | None = None,\n    ) -> "Resource":\n        return cls(\n            ResourceKind.SKILL,\n            name,\n            description,\n            content,\n            source,\n            related_resources=related_resources or {},\n        )\n\n    @classmethod\n    def prompt_template(\n        cls, name: str, content: str, *, source: str\n    ) -> "Resource":\n        return cls(\n            ResourceKind.PROMPT_TEMPLATE,\n            name,\n            f"Local Prompt Template {name}",\n            content,\n            source,\n        )\n\n    @classmethod\n    def extension_source(\n        cls, name: str, content: str, *, source: str\n    ) -> "Resource":\n        return cls(\n            ResourceKind.EXTENSION,\n            name,\n            f"Local Extension source {name}",\n            content,\n            source,\n        )\n\n    def __post_init__(self) -> None:\n        if self.kind in {\n            ResourceKind.SKILL,\n            ResourceKind.PROMPT_TEMPLATE,\n            ResourceKind.EXTENSION,\n        } and not _RESOURCE_NAME.fullmatch(self.name):\n            raise ValueError("resource names must be safe lowercase identifiers")\n        if not self.description.strip():\n            raise ValueError("resources require a description")\n        if not isinstance(self.content, str):\n            raise TypeError("resource content must be text")\n        if not self.source.strip():\n            raise ValueError("resources require a source")\n        if any(\n            not isinstance(name, str)\n            or not name\n            or Path(name).is_absolute()\n            or ".." in Path(name).parts\n            or not isinstance(content, str)\n            for name, content in self.related_resources.items()\n        ):\n            raise ValueError("related Skill resources require safe relative text paths")\n        object.__setattr__(\n            self,\n            "related_resources",\n            MappingProxyType(dict(sorted(self.related_resources.items()))),\n        )\n\n\n@dataclass(frozen=True, slots=True)\nclass ResourceEvidence:\n    kind: ResourceKind\n    scope: ResourceScope\n    source: str\n    sha256: str\n\n\n@dataclass(frozen=True, slots=True)\nclass LoadedResources:\n    skills: Mapping[str, Resource]\n    evidence: Mapping[str, ResourceEvidence]\n    context_files: tuple[Resource, ...] = ()\n    system_replacement: Resource | None = None\n    append_system: Resource | None = None\n    prompt_templates: Mapping[str, Resource] = field(default_factory=dict)\n    extensions: Mapping[str, Resource] = field(default_factory=dict)\n    settings: Resource | None = None\n    skipped_protected: tuple[str, ...] = ()\n\n    def __post_init__(self) -> None:\n        object.__setattr__(self, "skills", MappingProxyType(dict(self.skills)))\n        object.__setattr__(self, "evidence", MappingProxyType(dict(self.evidence)))\n        object.__setattr__(\n            self, "prompt_templates", MappingProxyType(dict(self.prompt_templates))\n        )\n        object.__setattr__(self, "extensions", MappingProxyType(dict(self.extensions)))\n\n    def expand_prompt(self, name: str, arguments: Mapping[str, str]) -> str:\n        try:\n            template = self.prompt_templates[name]\n        except KeyError:\n            raise KeyError(f"unknown Prompt Template: {name}") from None\n        if any(\n            not isinstance(key, str) or not isinstance(value, str)\n            for key, value in arguments.items()\n        ):\n            raise TypeError("Prompt Template arguments must map text names to text values")\n        try:\n            return template.content.format_map(dict(arguments))\n        except KeyError as error:\n            raise ValueError(\n                f"Prompt Template {name!r} requires argument {error.args[0]!r}"\n            ) from None\n\n\n@dataclass(frozen=True, slots=True)\nclass PromptAssembly:\n    text: str\n    prompt_hashes: Mapping[str, str]\n    resource_hashes: Mapping[str, str]\n    active_skills: tuple[str, ...]\n\n    def __post_init__(self) -> None:\n        object.__setattr__(\n            self, "prompt_hashes", MappingProxyType(dict(self.prompt_hashes))\n        )\n        object.__setattr__(\n            self, "resource_hashes", MappingProxyType(dict(self.resource_hashes))\n        )\n\n\nclass PromptAssembler:\n    """Assemble bounded system context in one documented deterministic order."""\n\n    def __init__(self, builtin_prompt: str) -> None:\n        if not builtin_prompt.strip():\n            raise ValueError("the built-in prompt cannot be empty")\n        self._builtin_prompt = builtin_prompt.strip()\n\n    def assemble(\n        self,\n        *,\n        tools: Sequence[Tool],\n        resources: LoadedResources,\n        active_skills: Sequence[str] = (),\n    ) -> PromptAssembly:\n        if resources.system_replacement is not None:\n            text = resources.system_replacement.content.strip() + "\\n"\n            digest = f"sha256:{sha256(text.encode(\'utf-8\')).hexdigest()}"\n            evidence = resources.evidence["system:replacement"]\n            return PromptAssembly(\n                text,\n                {"effective_system": digest},\n                {"system:replacement": evidence.sha256},\n                (),\n            )\n        accepted_skills = tuple(sorted(set(active_skills)))\n        unknown = [name for name in accepted_skills if name not in resources.skills]\n        if unknown:\n            raise KeyError(f"unknown Skill: {unknown[0]}")\n        sections = [self._builtin_prompt]\n        accepted_tools = sorted(tools, key=lambda tool: tool.name)\n        if accepted_tools:\n            sections.append(\n                "## Active Tools\\n"\n                + "\\n".join(\n                    f"- {tool.name}: {tool.description}" for tool in accepted_tools\n                )\n            )\n        if resources.skills:\n            sections.append(\n                "## Available Skills\\n"\n                + "\\n".join(\n                    f"- {skill.name}: {skill.description} ({skill.source})"\n                    for skill in sorted(\n                        resources.skills.values(), key=lambda resource: resource.name\n                    )\n                )\n            )\n        for name in accepted_skills:\n            skill = resources.skills[name]\n            sections.append(f"## Active Skill: {name}\\n{skill.content}")\n            sections.extend(\n                f"### Related Skill Resource: {relative}\\n{content}"\n                for relative, content in skill.related_resources.items()\n            )\n        sections.extend(\n            f"## Context File: {context.source}\\n{context.content}"\n            for context in resources.context_files\n        )\n        if resources.append_system is not None:\n            sections.append(\n                "## Appended System Instructions\\n"\n                + resources.append_system.content\n            )\n        text = "\\n\\n".join(section.strip() for section in sections) + "\\n"\n        digest = f"sha256:{sha256(text.encode(\'utf-8\')).hexdigest()}"\n        resource_hashes = {\n            name: evidence.sha256\n            for name, evidence in resources.evidence.items()\n            if ":related:" not in name\n        }\n        for name in accepted_skills:\n            for relative in resources.skills[name].related_resources:\n                key = f"skill:{name}:related:{relative}"\n                resource_hashes[key] = resources.evidence[key].sha256\n        return PromptAssembly(\n            text,\n            {"effective_system": digest},\n            resource_hashes,\n            accepted_skills,\n        )\n\n\nclass ProjectTrust(Protocol):\n    def is_trusted(self, canonical_workspace: Path) -> bool: ...\n\n\nclass MemoryProjectTrust:\n    """An explicit trust adapter useful to applications and deterministic tests."""\n\n    def __init__(self, trusted: Iterable[str | Path] = ()) -> None:\n        self._trusted = {Path(path).resolve() for path in trusted}\n\n    def is_trusted(self, canonical_workspace: Path) -> bool:\n        return canonical_workspace.resolve() in self._trusted\n\n    def approve(self, workspace: str | Path) -> None:\n        self._trusted.add(Path(workspace).resolve())\n\n\nclass JSONProjectTrustStore:\n    """Persist canonical Project Trust decisions in one transparent JSON file."""\n\n    def __init__(self, path: str | Path) -> None:\n        self._path = Path(path).resolve()\n\n    def is_trusted(self, canonical_workspace: Path) -> bool:\n        return str(canonical_workspace.resolve()) in self._read()\n\n    def approve(self, workspace: str | Path) -> None:\n        trusted = self._read()\n        trusted.add(str(Path(workspace).resolve()))\n        self._write(trusted)\n\n    def revoke(self, workspace: str | Path) -> None:\n        trusted = self._read()\n        trusted.discard(str(Path(workspace).resolve()))\n        self._write(trusted)\n\n    def _read(self) -> set[str]:\n        if not self._path.exists():\n            return set()\n        payload = json.loads(self._path.read_text(encoding="utf-8"))\n        if payload.get("schema_version") != 1:\n            raise ValueError(\n                "Unsupported Project Trust schema; migrate trusted-projects.json"\n            )\n        paths = payload.get("trusted_paths")\n        if not isinstance(paths, list) or any(\n            not isinstance(path, str) or not Path(path).is_absolute()\n            for path in paths\n        ):\n            raise ValueError("Project Trust file contains invalid canonical paths")\n        return set(paths)\n\n    def _write(self, trusted: set[str]) -> None:\n        self._path.parent.mkdir(parents=True, exist_ok=True)\n        payload = json.dumps(\n            {"schema_version": 1, "trusted_paths": sorted(trusted)},\n            ensure_ascii=False,\n            indent=2,\n            sort_keys=True,\n        ) + "\\n"\n        descriptor, temporary_name = tempfile.mkstemp(\n            prefix=f".{self._path.name}.",\n            suffix=".omega-tmp",\n            dir=self._path.parent,\n        )\n        temporary = Path(temporary_name)\n        try:\n            with os.fdopen(descriptor, "w", encoding="utf-8") as stream:\n                stream.write(payload)\n                stream.flush()\n                os.fsync(stream.fileno())\n            os.replace(temporary, self._path)\n        except BaseException:\n            temporary.unlink(missing_ok=True)\n            raise\n\n\nclass ResourceLoader:\n    """Resolve local resource scopes before constructing reusable harness values."""\n\n    def __init__(\n        self,\n        workspace: str | Path,\n        *,\n        working_directory: str | Path | None = None,\n        user_root: str | Path | None = None,\n        trust: ProjectTrust | None = None,\n        approve_project: bool = False,\n        context_enabled: bool = True,\n        explicit: Sequence[Resource] = (),\n        builtins: Sequence[Resource] = (),\n    ) -> None:\n        self._workspace = Path(workspace).resolve()\n        if not self._workspace.is_dir():\n            raise ValueError("workspace must be an existing directory")\n        self._working_directory = (\n            Path(working_directory).resolve()\n            if working_directory is not None\n            else self._workspace\n        )\n        if not self._working_directory.is_relative_to(self._workspace):\n            raise ValueError("working_directory must be inside the workspace")\n        self._user_root = Path(user_root).resolve() if user_root is not None else None\n        self._trust = trust\n        self._approve_project = approve_project\n        self._context_enabled = context_enabled\n        self._explicit = tuple(explicit)\n        self._builtins = tuple(builtins)\n\n    @property\n    def workspace(self) -> Path:\n        return self._workspace\n\n    def load(self) -> LoadedResources:\n        resolved_by_kind: dict[ResourceKind, dict[str, Resource]] = {\n            ResourceKind.SKILL: {},\n            ResourceKind.PROMPT_TEMPLATE: {},\n            ResourceKind.EXTENSION: {},\n        }\n        sources = (\n            (ResourceScope.BUILTIN, self._builtins),\n            (ResourceScope.USER, self._discover_scoped(self._user_root)),\n            (\n                ResourceScope.PROJECT,\n                self._discover_scoped(self._workspace / ".omega")\n                if self._project_is_trusted()\n                else (),\n            ),\n            (ResourceScope.EXPLICIT, self._explicit),\n        )\n        for scope, scoped_resources in sources:\n            for resource in scoped_resources:\n                target = resolved_by_kind.get(resource.kind)\n                if target is not None:\n                    target[resource.name] = replace(resource, scope=scope)\n        resolved = resolved_by_kind[ResourceKind.SKILL]\n        prompt_templates = resolved_by_kind[ResourceKind.PROMPT_TEMPLATE]\n        extensions = resolved_by_kind[ResourceKind.EXTENSION]\n        context_files = self._discover_context_files() if self._context_enabled else ()\n        system_replacement: Resource | None = None\n        append_system: Resource | None = None\n        settings: Resource | None = None\n        skipped: tuple[str, ...] = ()\n        if self._project_is_trusted():\n            system_replacement = self._read_optional_project_instruction(\n                "SYSTEM.md", ResourceKind.SYSTEM\n            )\n            append_system = self._read_optional_project_instruction(\n                "APPEND_SYSTEM.md", ResourceKind.APPEND_SYSTEM\n            )\n            settings = self._read_optional_project_instruction(\n                "settings.json", ResourceKind.SETTINGS\n            )\n            if settings is not None:\n                try:\n                    parsed_settings = json.loads(settings.content)\n                except json.JSONDecodeError as error:\n                    raise ValueError(\n                        "Project settings.json must contain valid JSON"\n                    ) from error\n                if not isinstance(parsed_settings, dict):\n                    raise ValueError("Project settings.json must contain a JSON object")\n        else:\n            skipped = self._protected_project_paths()\n        evidence = {\n            f"skill:{name}": ResourceEvidence(\n                resource.kind,\n                resource.scope,\n                resource.source,\n                f"sha256:{sha256(resource.content.encode(\'utf-8\')).hexdigest()}",\n            )\n            for name, resource in sorted(resolved.items())\n        }\n        for name, resource in sorted(resolved.items()):\n            source_root = Path(resource.source).parent\n            for relative, content in resource.related_resources.items():\n                related = replace(\n                    resource,\n                    content=content,\n                    source=str((source_root / relative).resolve()),\n                    related_resources={},\n                )\n                evidence[f"skill:{name}:related:{relative}"] = self._evidence(related)\n        for prefix, named_resources in (\n            ("prompt", prompt_templates),\n            ("extension", extensions),\n        ):\n            for name, resource in sorted(named_resources.items()):\n                evidence[f"{prefix}:{name}"] = self._evidence(resource)\n        for index, resource in enumerate(context_files):\n            evidence[f"context:{index}:{resource.source}"] = self._evidence(resource)\n        if system_replacement is not None:\n            evidence["system:replacement"] = self._evidence(system_replacement)\n        if append_system is not None:\n            evidence["system:append"] = self._evidence(append_system)\n        if settings is not None:\n            evidence["settings:project"] = self._evidence(settings)\n        return LoadedResources(\n            resolved,\n            evidence,\n            context_files,\n            system_replacement,\n            append_system,\n            prompt_templates,\n            extensions,\n            settings=settings,\n            skipped_protected=skipped,\n        )\n\n    def _project_is_trusted(self) -> bool:\n        return self._approve_project or (\n            self._trust is not None and self._trust.is_trusted(self._workspace)\n        )\n\n    @staticmethod\n    def _evidence(resource: Resource) -> ResourceEvidence:\n        return ResourceEvidence(\n            resource.kind,\n            resource.scope,\n            resource.source,\n            f"sha256:{sha256(resource.content.encode(\'utf-8\')).hexdigest()}",\n        )\n\n    def _discover_context_files(self) -> tuple[Resource, ...]:\n        relative = self._working_directory.relative_to(self._workspace)\n        directories = [self._workspace]\n        current = self._workspace\n        for part in relative.parts:\n            current = current / part\n            directories.append(current)\n        contexts: list[Resource] = []\n        for directory in directories:\n            path = directory / "AGENTS.md"\n            if path.is_file():\n                contexts.append(\n                    Resource(\n                        ResourceKind.CONTEXT,\n                        path.parent.relative_to(self._workspace).as_posix() or ".",\n                        "Hierarchical project context",\n                        self._read_scoped_text(path, self._workspace),\n                        str(path.resolve()),\n                        ResourceScope.PROJECT,\n                    )\n                )\n        return tuple(contexts)\n\n    def _read_optional_project_instruction(\n        self, name: str, kind: ResourceKind\n    ) -> Resource | None:\n        path = self._workspace / ".omega" / name\n        if not path.is_file():\n            return None\n        return Resource(\n            kind,\n            name,\n            f"Trusted project {name}",\n            self._read_scoped_text(path, self._workspace),\n            str(path.resolve()),\n            ResourceScope.PROJECT,\n        )\n\n    def _protected_project_paths(self) -> tuple[str, ...]:\n        root = self._workspace / ".omega"\n        candidates = [\n            *(root / "skills").glob("*/SKILL.md"),\n            *(root / "prompts").glob("*.md"),\n            *(root / "extensions").glob("*.py"),\n            root / "SYSTEM.md",\n            root / "APPEND_SYSTEM.md",\n            root / "settings.json",\n        ]\n        return tuple(\n            str(path.resolve()) for path in sorted(candidates) if path.is_file()\n        )\n\n    @staticmethod\n    def _discover_scoped(root: Path | None) -> tuple[Resource, ...]:\n        if root is None:\n            return ()\n        skill_root = root / "skills"\n        resources: list[Resource] = []\n        if skill_root.is_dir():\n            for path in sorted(skill_root.glob("*/SKILL.md")):\n                resources.append(ResourceLoader._read_skill(path, root))\n        for path in sorted((root / "prompts").glob("*.md")):\n            resources.append(\n                Resource.prompt_template(\n                    path.stem,\n                    ResourceLoader._read_scoped_text(path, root),\n                    source=str(path.resolve()),\n                )\n            )\n        for path in sorted((root / "extensions").glob("*.py")):\n            resources.append(\n                Resource.extension_source(\n                    path.stem,\n                    ResourceLoader._read_scoped_text(path, root),\n                    source=str(path.resolve()),\n                )\n            )\n        return tuple(resources)\n\n    @staticmethod\n    def _read_scoped_text(path: Path, scope_root: Path) -> str:\n        resolved = path.resolve()\n        if not resolved.is_relative_to(scope_root.resolve()):\n            raise ValueError(f"local resource escapes its scope: {path}")\n        return resolved.read_text(encoding="utf-8")\n\n    @staticmethod\n    def _read_skill(path: Path, scope_root: Path) -> Resource:\n        text = ResourceLoader._read_scoped_text(path, scope_root)\n        if not text.startswith("---\\n"):\n            raise ValueError(f"Skill {path} requires YAML-style front matter")\n        try:\n            header, content = text[4:].split("\\n---\\n", 1)\n        except ValueError as error:\n            raise ValueError(f"Skill {path} has unterminated front matter") from error\n        metadata: dict[str, str] = {}\n        for line in header.splitlines():\n            key, separator, value = line.partition(":")\n            if separator:\n                metadata[key.strip()] = value.strip()\n        name = metadata.get("name", "")\n        description = metadata.get("description", "")\n        root = path.parent.resolve()\n        related: dict[str, str] = {}\n        for candidate in sorted(path.parent.rglob("*")):\n            if not candidate.is_file() or candidate == path:\n                continue\n            resolved = candidate.resolve()\n            if not resolved.is_relative_to(root):\n                raise ValueError(f"Skill resource escapes its directory: {candidate}")\n            raw = resolved.read_bytes()\n            if b"\\x00" in raw:\n                raise ValueError(f"Skill related resource must be text: {candidate}")\n            try:\n                related[resolved.relative_to(root).as_posix()] = raw.decode("utf-8")\n            except UnicodeDecodeError as error:\n                raise ValueError(\n                    f"Skill related resource must be UTF-8 text: {candidate}"\n                ) from error\n        return Resource.skill(\n            name,\n            description,\n            content.strip(),\n            source=str(path.resolve()),\n            related_resources=related,\n        )\n'


In [ ]:
CODING_SOURCE = '"""Optional host-authority Coding Tool Preset."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass\nfrom hashlib import sha256\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport signal\nimport stat\nimport subprocess\nimport tempfile\nfrom typing import Protocol\n\nfrom .tools import CompleteOutputReference, Tool, ToolErrorCode, ToolResult\nfrom .tools import ProcessResult, TruncationDirection\nfrom .model import AgentMessage, ModelAdapter, ModelSpec, Role\nfrom .resources import LoadedResources, PromptAssembler, PromptAssembly, ResourceLoader\nfrom .runtime import AgentRuntime\nfrom .session import AgentSession\n\n\n_SENSITIVE_ENVIRONMENT = re.compile(\n    r"(?:api[_-]?key|access[_-]?key|token|secret|password|passwd|"\n    r"authorization|credential|private[_-]?key)",\n    re.IGNORECASE,\n)\n\n\nclass WorkspaceBoundaryError(ValueError):\n    """A file Tool path could not be confined to its Workspace Boundary."""\n\n\nclass ArtifactStore(Protocol):\n    def put_text(self, content: str) -> str: ...\n\n\nclass FileArtifactStore:\n    """Retain sanitized text under a content-addressed local reference."""\n\n    def __init__(\n        self,\n        root: str | Path,\n        *,\n        redact: Callable[[str], str] | None = None,\n    ) -> None:\n        self._root = Path(root).resolve()\n        self._root.mkdir(parents=True, exist_ok=True)\n        self._redact = redact or (lambda text: text)\n\n    def put_text(self, content: str) -> str:\n        sanitized = self._redact(content)\n        if not isinstance(sanitized, str):\n            raise TypeError("Artifact redactor must return text")\n        digest = sha256(sanitized.encode("utf-8")).hexdigest()\n        path = self._root / "sha256" / f"{digest}.txt"\n        if not path.exists():\n            _atomic_write(path, sanitized)\n        return f"sha256:{digest}"\n\n    def read_text(self, reference: str) -> str:\n        prefix, separator, digest = reference.partition(":")\n        if prefix != "sha256" or not separator or not re.fullmatch(\n            r"[0-9a-f]{64}", digest\n        ):\n            raise ValueError("invalid Artifact reference")\n        return (self._root / "sha256" / f"{digest}.txt").read_text(\n            encoding="utf-8"\n        )\n\n\nclass WorkspaceBoundary:\n    """Resolve file Tool paths through canonical parents inside one workspace."""\n\n    def __init__(self, root: str | Path) -> None:\n        self._root = Path(root).resolve()\n        if not self._root.is_dir():\n            raise ValueError("Workspace Boundary must be an existing directory")\n\n    @property\n    def root(self) -> Path:\n        return self._root\n\n    def resolve(self, supplied: object) -> Path:\n        if not isinstance(supplied, str) or not supplied:\n            raise WorkspaceBoundaryError("file path must be a non-empty string")\n        path = Path(supplied)\n        candidate = path if path.is_absolute() else self._root / path\n        try:\n            resolved = candidate.resolve(strict=False)\n        except (OSError, RuntimeError) as error:\n            raise WorkspaceBoundaryError(\n                "path cannot be resolved inside the Workspace Boundary"\n            ) from error\n        if not resolved.is_relative_to(self._root):\n            raise WorkspaceBoundaryError(\n                "path resolves outside the Workspace Boundary"\n            )\n        return resolved\n\n\ndef filter_sensitive_environment(\n    environment: Mapping[str, str],\n    *,\n    allow_sensitive: Sequence[str] = (),\n) -> dict[str, str]:\n    """Remove credential-shaped names unless the caller deliberately allows one."""\n\n    allowed = {name.casefold() for name in allow_sensitive}\n    return {\n        name: value\n        for name, value in environment.items()\n        if name.casefold() in allowed or _SENSITIVE_ENVIRONMENT.search(name) is None\n    }\n\n\ndef resolve_bash(\n    shell_path: str | Path | None = None,\n    *,\n    environment: Mapping[str, str] | None = None,\n) -> Path:\n    """Resolve one real Bash executable without translating shell commands."""\n\n    if shell_path is not None:\n        candidate = Path(shell_path).expanduser().resolve()\n        if candidate.is_file() and (\n            os.name == "nt" or os.access(candidate, os.X_OK)\n        ):\n            return candidate\n        raise FileNotFoundError(\n            f"configured Bash executable does not exist or is not executable: {candidate}"\n        )\n    source = dict(os.environ if environment is None else environment)\n    candidates: list[Path] = []\n    if os.name == "nt":\n        for variable in ("ProgramFiles", "ProgramFiles(x86)", "LocalAppData"):\n            base = source.get(variable)\n            if base:\n                candidates.append(Path(base) / "Git" / "bin" / "bash.exe")\n                if variable == "LocalAppData":\n                    candidates.append(\n                        Path(base) / "Programs" / "Git" / "bin" / "bash.exe"\n                    )\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate.resolve()\n    found = shutil.which("bash", path=source.get("PATH"))\n    if found is not None:\n        return Path(found).resolve()\n    raise FileNotFoundError(\n        "A real Bash executable is required; install Bash (Git Bash on Windows) "\n        "or configure shell_path"\n    )\n\n\nclass BashOperations:\n    """Execute real Bash commands with fixed cwd and cancellable host authority."""\n\n    def __init__(\n        self,\n        workspace: str | Path,\n        *,\n        shell_path: str | Path | None = None,\n        environment: Mapping[str, str] | None = None,\n        allow_sensitive: Sequence[str] = (),\n        default_timeout_seconds: float = 60.0,\n    ) -> None:\n        self._workspace = Path(workspace).resolve()\n        if not self._workspace.is_dir():\n            raise ValueError("Bash workspace must be an existing directory")\n        if default_timeout_seconds <= 0:\n            raise ValueError("default Bash timeout must be positive")\n        source = dict(os.environ if environment is None else environment)\n        self._shell = resolve_bash(shell_path, environment=source)\n        self._environment = filter_sensitive_environment(\n            source, allow_sensitive=allow_sensitive\n        )\n        self._default_timeout_seconds = default_timeout_seconds\n\n    @property\n    def shell_path(self) -> Path:\n        return self._shell\n\n    @property\n    def workspace(self) -> Path:\n        return self._workspace\n\n    async def run(\n        self,\n        command: str,\n        *,\n        timeout_seconds: float | None = None,\n    ) -> ProcessResult:\n        if not isinstance(command, str) or not command.strip():\n            raise ValueError("Bash command must be non-empty text")\n        timeout = (\n            self._default_timeout_seconds\n            if timeout_seconds is None\n            else timeout_seconds\n        )\n        if timeout <= 0:\n            raise ValueError("Bash timeout must be positive")\n        if os.name == "nt":\n            process = await asyncio.create_subprocess_exec(\n                str(self._shell),\n                "--noprofile",\n                "--norc",\n                "-lc",\n                command,\n                cwd=self._workspace,\n                env=self._environment,\n                stdout=asyncio.subprocess.PIPE,\n                stderr=asyncio.subprocess.PIPE,\n                creationflags=getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0),\n            )\n        else:\n            process = await asyncio.create_subprocess_exec(\n                str(self._shell),\n                "--noprofile",\n                "--norc",\n                "-lc",\n                command,\n                cwd=self._workspace,\n                env=self._environment,\n                stdout=asyncio.subprocess.PIPE,\n                stderr=asyncio.subprocess.PIPE,\n                start_new_session=True,\n            )\n        try:\n            stdout, stderr = await asyncio.wait_for(\n                process.communicate(), timeout=timeout\n            )\n        except asyncio.CancelledError:\n            await self._terminate(process)\n            raise\n        except TimeoutError:\n            await self._terminate(process)\n            raise TimeoutError(f"Bash command exceeded {timeout:g} seconds") from None\n        assert process.returncode is not None\n        return ProcessResult(\n            process.returncode,\n            stdout.decode("utf-8", errors="replace"),\n            stderr.decode("utf-8", errors="replace"),\n        )\n\n    @staticmethod\n    async def _terminate(process: asyncio.subprocess.Process) -> None:\n        if process.returncode is not None:\n            return\n        if os.name == "nt":\n            process.terminate()\n        else:\n            try:\n                os.killpg(process.pid, signal.SIGTERM)\n            except ProcessLookupError:\n                return\n        try:\n            await asyncio.wait_for(process.wait(), timeout=1)\n        except TimeoutError:\n            if os.name == "nt":\n                process.kill()\n            else:\n                try:\n                    os.killpg(process.pid, signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n            await process.wait()\n\n\ndef _tool_error(tool_name: str, error: Exception) -> ToolResult:\n    return ToolResult(\n        f"Tool error [execution_failed] for \'{tool_name}\': {error}",\n        is_error=True,\n        error_code=ToolErrorCode.EXECUTION_FAILED,\n    )\n\n\ndef _atomic_write(path: Path, content: str) -> None:\n    existing_mode = stat.S_IMODE(path.stat().st_mode) if path.exists() else None\n    path.parent.mkdir(parents=True, exist_ok=True)\n    descriptor, temporary_name = tempfile.mkstemp(\n        prefix=f".{path.name}.", suffix=".omega-tmp", dir=path.parent\n    )\n    temporary = Path(temporary_name)\n    try:\n        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as stream:\n            stream.write(content)\n            stream.flush()\n            os.fsync(stream.fileno())\n        if existing_mode is not None:\n            os.chmod(temporary, existing_mode)\n        os.replace(temporary, path)\n    except BaseException:\n        temporary.unlink(missing_ok=True)\n        raise\n\n\ndef _read_text_for_mutation(path: Path) -> str:\n    raw = path.read_bytes()\n    if b"\\x00" in raw:\n        raise ValueError("binary files cannot be mutated by text Tools")\n    try:\n        return raw.decode("utf-8")\n    except UnicodeDecodeError as error:\n        raise ValueError("binary or non-UTF-8 files cannot be mutated") from error\n\n\n@dataclass(frozen=True, slots=True)\nclass CodingToolPreset:\n    workspace: WorkspaceBoundary\n    bash_operations: BashOperations\n    tools: tuple[Tool, ...]\n\n    def tool(self, name: str) -> Tool:\n        for tool in self.tools:\n            if tool.name == name:\n                return tool\n        raise KeyError(f"unknown Coding Tool: {name}")\n\n\n@dataclass(frozen=True, slots=True)\nclass CodingAgent:\n    """The assembled Coding Agent product around one reusable AgentSession."""\n\n    session: AgentSession\n    preset: CodingToolPreset\n    resources: LoadedResources\n    prompt: PromptAssembly\n\n    @property\n    def startup_evidence(self) -> tuple[tuple[str, str], ...]:\n        return tuple(sorted(self.prompt.resource_hashes.items()))\n\n\ndef create_coding_tool_preset(\n    workspace: str | Path,\n    *,\n    bash_operations: BashOperations | None = None,\n    shell_path: str | Path | None = None,\n    environment: Mapping[str, str] | None = None,\n    allow_sensitive_environment: Sequence[str] = (),\n    artifact_store: ArtifactStore | None = None,\n    artifact_threshold_bytes: int = 50 * 1024,\n) -> CodingToolPreset:\n    """Create explicitly installed Coding Tools; no sandbox is implied."""\n\n    boundary = WorkspaceBoundary(workspace)\n    if artifact_threshold_bytes <= 0:\n        raise ValueError("artifact_threshold_bytes must be positive")\n    bash_backend = bash_operations or BashOperations(\n        boundary.root,\n        shell_path=shell_path,\n        environment=environment,\n        allow_sensitive=allow_sensitive_environment,\n    )\n    if bash_backend.workspace != boundary.root:\n        raise ValueError("BashOperations must use the Coding Tool Preset workspace")\n\n    def complete_output(content: str) -> CompleteOutputReference | None:\n        if (\n            artifact_store is None\n            or len(content.encode("utf-8")) <= artifact_threshold_bytes\n        ):\n            return None\n        return CompleteOutputReference.artifact(artifact_store.put_text(content))\n\n    async def read(arguments: dict[str, object]) -> ToolResult:\n        try:\n            path = boundary.resolve(arguments.get("path"))\n            if not path.is_file():\n                raise ValueError("read path must be an existing text file")\n            raw = path.read_bytes()\n            if b"\\x00" in raw:\n                raise ValueError("binary files cannot be read as text")\n            try:\n                content = raw.decode("utf-8")\n            except UnicodeDecodeError as error:\n                raise ValueError(\n                    "binary or non-UTF-8 files cannot be read as text"\n                ) from error\n            offset = arguments.get("offset", 1)\n            limit = arguments.get("limit")\n            if not isinstance(offset, int) or isinstance(offset, bool) or offset < 1:\n                raise ValueError("read offset must be a positive line number")\n            if limit is not None and (\n                not isinstance(limit, int) or isinstance(limit, bool) or limit < 1\n            ):\n                raise ValueError("read limit must be a positive line count")\n            lines = content.splitlines(keepends=True)\n            selected = (\n                lines[offset - 1 :]\n                if limit is None\n                else lines[offset - 1 : offset - 1 + limit]\n            )\n            return ToolResult(\n                "".join(selected),\n                metadata={\n                    "path": path.relative_to(boundary.root).as_posix(),\n                    "offset": offset,\n                    "lines": len(selected),\n                },\n                complete_output=complete_output(content),\n            )\n        except (OSError, ValueError) as error:\n            return _tool_error("read", error)\n\n    async def write(arguments: dict[str, object]) -> ToolResult:\n        try:\n            path = boundary.resolve(arguments.get("path"))\n            content = arguments.get("content")\n            if not isinstance(content, str):\n                raise ValueError("write content must be text")\n            if path.exists():\n                if not path.is_file():\n                    raise ValueError("write path must be a text file")\n                _read_text_for_mutation(path)\n            _atomic_write(path, content)\n            return ToolResult(\n                f"Wrote {len(content.encode(\'utf-8\'))} bytes to "\n                f"{path.relative_to(boundary.root).as_posix()}",\n                metadata={"path": path.relative_to(boundary.root).as_posix()},\n            )\n        except (OSError, ValueError) as error:\n            return _tool_error("write", error)\n\n    async def edit(arguments: dict[str, object]) -> ToolResult:\n        try:\n            path = boundary.resolve(arguments.get("path"))\n            if not path.is_file():\n                raise ValueError("edit path must be an existing text file")\n            old_text = arguments.get("old_text")\n            new_text = arguments.get("new_text")\n            if not isinstance(old_text, str) or not old_text:\n                raise ValueError("edit old_text must be non-empty text")\n            if not isinstance(new_text, str):\n                raise ValueError("edit new_text must be text")\n            content = _read_text_for_mutation(path)\n            occurrences = content.count(old_text)\n            if occurrences != 1:\n                raise ValueError(\n                    f"edit old_text must match exactly once; found {occurrences} matches"\n                )\n            updated = content.replace(old_text, new_text, 1)\n            _atomic_write(path, updated)\n            relative = path.relative_to(boundary.root).as_posix()\n            return ToolResult(\n                f"Edited {relative}",\n                metadata={"path": relative, "replacements": 1},\n            )\n        except (OSError, ValueError) as error:\n            return _tool_error("edit", error)\n\n    async def bash(arguments: dict[str, object]) -> ToolResult:\n        try:\n            command = arguments.get("command")\n            timeout = arguments.get("timeout_seconds")\n            if not isinstance(command, str):\n                raise ValueError("Bash command must be text")\n            if timeout is not None and (\n                not isinstance(timeout, (int, float))\n                or isinstance(timeout, bool)\n                or timeout <= 0\n            ):\n                raise ValueError("Bash timeout_seconds must be positive")\n            result = await bash_backend.run(command, timeout_seconds=timeout)\n            content = result.stdout\n            if result.stderr:\n                content += ("\\n" if content else "") + result.stderr\n            return ToolResult(\n                content,\n                metadata={"returncode": result.returncode},\n                is_error=result.returncode != 0,\n                error_code=(\n                    ToolErrorCode.EXECUTION_FAILED if result.returncode != 0 else None\n                ),\n                complete_output=complete_output(content),\n            )\n        except (OSError, TimeoutError, ValueError) as error:\n            return _tool_error("bash", error)\n\n    read_tool = Tool(\n        "read",\n        "Read UTF-8 text by line range inside the Workspace Boundary",\n        {\n            "type": "object",\n            "properties": {\n                "path": {"type": "string"},\n                "offset": {"type": "integer", "minimum": 1},\n                "limit": {"type": "integer", "minimum": 1},\n            },\n            "required": ["path"],\n            "additionalProperties": False,\n        },\n        read,\n    )\n    write_tool = Tool(\n        "write",\n        "Atomically write one UTF-8 text file inside the Workspace Boundary",\n        {\n            "type": "object",\n            "properties": {\n                "path": {"type": "string"},\n                "content": {"type": "string"},\n            },\n            "required": ["path", "content"],\n            "additionalProperties": False,\n        },\n        write,\n        sequential=True,\n    )\n    edit_tool = Tool(\n        "edit",\n        "Atomically replace one exact text occurrence inside the Workspace Boundary",\n        {\n            "type": "object",\n            "properties": {\n                "path": {"type": "string"},\n                "old_text": {"type": "string", "minLength": 1},\n                "new_text": {"type": "string"},\n            },\n            "required": ["path", "old_text", "new_text"],\n            "additionalProperties": False,\n        },\n        edit,\n        sequential=True,\n    )\n    bash_tool = Tool(\n        "bash",\n        "Run a real Bash command in the fixed workspace with host-process authority",\n        {\n            "type": "object",\n            "properties": {\n                "command": {"type": "string", "minLength": 1},\n                "timeout_seconds": {"type": "number", "exclusiveMinimum": 0},\n            },\n            "required": ["command"],\n            "additionalProperties": False,\n        },\n        bash,\n        sequential=True,\n        output_direction=TruncationDirection.TAIL,\n    )\n    return CodingToolPreset(\n        boundary,\n        bash_backend,\n        (read_tool, write_tool, edit_tool, bash_tool),\n    )\n\n\ndef create_coding_agent(\n    adapter: ModelAdapter,\n    model: ModelSpec,\n    workspace: str | Path,\n    *,\n    resource_loader: ResourceLoader | None = None,\n    prompt_assembler: PromptAssembler | None = None,\n    active_skills: Sequence[str] = (),\n    preset: CodingToolPreset | None = None,\n) -> CodingAgent:\n    """Assemble local resources and explicitly install the Coding Tool Preset."""\n\n    boundary = Path(workspace).resolve()\n    effective_preset = preset or create_coding_tool_preset(boundary)\n    if effective_preset.workspace.root != boundary:\n        raise ValueError("Coding Tool Preset must use the Coding Agent workspace")\n    effective_loader = resource_loader or ResourceLoader(boundary)\n    if effective_loader.workspace != boundary:\n        raise ValueError("ResourceLoader must use the Coding Agent workspace")\n    resources = effective_loader.load()\n    assembler = prompt_assembler or PromptAssembler(\n        "You are Omega, a coding agent. Work carefully inside the supplied "\n        "workspace, use Tools for evidence, and verify changes before finishing."\n    )\n    prompt = assembler.assemble(\n        tools=effective_preset.tools,\n        resources=resources,\n        active_skills=active_skills,\n    )\n    runtime = AgentRuntime(\n        adapter,\n        model,\n        tools=effective_preset.tools,\n        history=(AgentMessage.text(Role.SYSTEM, prompt.text),),\n    )\n    session = AgentSession(\n        runtime,\n        prompt_hashes=prompt.prompt_hashes,\n        resource_hashes=prompt.resource_hashes,\n    )\n    return CodingAgent(session, effective_preset, resources, prompt)\n'


In [ ]:
INIT_SOURCE = 'from .compaction import (\n    CharacterTokenEstimator,\n    CompactionCheckpoint,\n    CompactionPlan,\n    CompactionPolicy,\n    CompactionStrategy,\n    CompactionTrigger,\n    CompactionWarning,\n    CompactionWarningCode,\n    ResolvedCompactionPolicy,\n    StructuredSummary,\n    TokenEstimator,\n)\nfrom .model import (\n    AgentMessage,\n    ContentBlock,\n    ModelAdapter,\n    ModelAdapterError,\n    ModelEnd,\n    ModelError,\n    ModelErrorCode,\n    ModelEvent,\n    ModelMessage,\n    ModelOperation,\n    ModelProtocolError,\n    ModelRequest,\n    ModelResult,\n    ModelSpec,\n    ModelToolResultMessage,\n    OpenAICompatibleAdapter,\n    OpenAICompatibleConfig,\n    Role,\n    ScriptedModelAdapter,\n    StopReason,\n    TextContent,\n    TextDelta,\n    ToolCallContent,\n    ToolCallDelta,\n    UnsupportedContentError,\n    Usage,\n    UsageUpdate,\n    complete,\n    to_model_messages,\n)\nfrom .extensions import (\n    Extension,\n    ExtensionAPI,\n    ExtensionHost,\n    ExtensionInitializationError,\n    SubscriberRegistration,\n    HookContext,\n    HookExecutionError,\n    HookPoint,\n    HookRegistration,\n    CompactionHookRequest,\n    CommandExecutionError,\n    CommandRegistration,\n    CustomEntry,\n    RunAnnotation,\n    LifecycleWarning,\n    ReloadResult,\n    ReplacementRecord,\n)\nfrom .persistence import (\n    ConversationMessage,\n    JSONLSessionStore,\n    JSONLSessionWriter,\n    MemorySessionStore,\n    MemorySessionWriter,\n    MigrationResult,\n    RecoveryCode,\n    RecoveryWarning,\n    SESSION_SCHEMA_VERSION,\n    SchemaVersion,\n    SessionBusyError,\n    SessionEntry,\n    SessionStore,\n    SessionState,\n    SessionWriter,\n    UnsupportedSchemaVersionError,\n    migrate_session_file,\n)\nfrom .runtime import (\n    AgentRunHandle,\n    AgentRuntime,\n    AssistantOutcome,\n    EventType,\n    RetryPolicy,\n    RunSnapshot,\n    RunGuard,\n    RuntimeEvent,\n    Sleeper,\n    ContextOverflowRecovery,\n    SettledTurnHandler,\n    SummaryGeneration,\n    TerminalStatus,\n    TurnInput,\n)\nfrom .session import (\n    AgentSession,\n    CompactionResult,\n    InputKind,\n    PendingInput,\n    SessionRunHandle,\n    SessionRunResult,\n    create_agent_session,\n)\nfrom .tools import (\n    AsyncioProcessOperations,\n    CompleteOutputKind,\n    CompleteOutputReference,\n    LocalToolExecutor,\n    PreparedToolCall,\n    ProcessResult,\n    Tool,\n    ToolErrorCode,\n    ToolExecutor,\n    ToolOutputBudget,\n    ToolResult,\n    ToolResultMessage,\n    TruncationDirection,\n    TruncationNotice,\n)\nfrom .resources import (\n    LoadedResources,\n    JSONProjectTrustStore,\n    MemoryProjectTrust,\n    PromptAssembler,\n    PromptAssembly,\n    ProjectTrust,\n    Resource,\n    ResourceEvidence,\n    ResourceKind,\n    ResourceLoader,\n    ResourceScope,\n)\nfrom .coding import (\n    ArtifactStore,\n    BashOperations,\n    CodingAgent,\n    CodingToolPreset,\n    FileArtifactStore,\n    WorkspaceBoundary,\n    WorkspaceBoundaryError,\n    create_coding_agent,\n    create_coding_tool_preset,\n    filter_sensitive_environment,\n    resolve_bash,\n)\n\n__all__ = [name for name in globals() if not name.startswith("_")]\n'


In [ ]:
import asyncio
import importlib
import shutil
from tempfile import TemporaryDirectory

temporary_package = TemporaryDirectory(prefix='chapter-08-minimal-')
package = Path(temporary_package.name) / 'agent_harness'
shutil.copytree(CHAPTER_7 / 'src' / 'agent_harness', package)
for name, text in {
    'resources.py': RESOURCES_SOURCE,
    'coding.py': CODING_SOURCE,
    '__init__.py': INIT_SOURCE,
}.items():
    (package / name).write_text(text, encoding='utf-8')
sys.path.insert(0, temporary_package.name)
chapter8 = importlib.import_module('agent_harness')

temporary_workspace = TemporaryDirectory(prefix='chapter-08-workspace-')
workspace = Path(temporary_workspace.name)
(workspace / 'AGENTS.md').write_text('Keep changes verified.', encoding='utf-8')
resources = chapter8.ResourceLoader(workspace).load()
preset = chapter8.create_coding_tool_preset(workspace)

# Windows Jupyter kernels use a Selector loop that cannot launch subprocesses.
def run_bash_demo():
    return asyncio.run(preset.tool('bash').execute({'command': "printf 'local bash'"}))

bash_result = await asyncio.to_thread(run_bash_demo)
assert bash_result.content == 'local bash'
assert [tool.name for tool in preset.tools] == ['read', 'write', 'edit', 'bash']
assert len(resources.context_files) == 1


## Staged Construction

The cumulative package carries every Chapter 7 module and regression test forward. These Export Cells add behavior tests, the Chapter contract, package metadata, and user-facing documentation.

In [ ]:
'[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "agent-harness"\nversion = "0.8.0"\ndescription = "Chapter 8 trusted local resources and optional Coding Agent Tools"\nreadme = "README.md"\nrequires-python = ">=3.11"\ndependencies = ["jsonschema>=4.23,<5", "openai>=1.40,<3"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.setuptools.package-data]\nagent_harness = ["py.typed"]\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n'

In [ ]:
'# Agent Harness — Chapter 8 Checkpoint\n\nThis cumulative Checkpoint adds the optional Coding Agent product layer. `ResourceLoader` resolves only explicit and local project, user, and built-in resources; Project Trust is recorded against a canonical workspace; `PromptAssembler` keeps system context deterministic and Skill instructions progressively disclosed.\n\n`create_coding_tool_preset()` explicitly installs `read`, `write`, `edit`, and `bash`. File Tools enforce a canonical Workspace Boundary, reject binary mutation, and atomically replace text. Bash is a real host process with a fixed working directory, sensitive-environment filtering, cancellation, timeout, bounded model output through the Runtime, and optional sanitized content-addressed Artifacts. Neither Bash nor Extensions are a sandbox guarantee.\n\nOffline tests use fixture Context Files and resources, `ScriptedModelAdapter`, a real local Bash executable, and a loopback OpenAI-compatible endpoint. They do not require a repository-root `AGENTS.md`. The optional real-endpoint smoke test remains skipped unless its documented endpoint, API key, and model variables are deliberately configured.\n'

In [ ]:
TEST_RESOURCES_SOURCE = 'from __future__ import annotations\n\nfrom pathlib import Path\n\nimport pytest\n\nfrom agent_harness import (\n    JSONProjectTrustStore,\n    MemoryProjectTrust,\n    PromptAssembler,\n    Resource,\n    ResourceLoader,\n    ResourceScope,\n    Tool,\n    ToolResult,\n)\n\n\ndef _write_skill(root: Path, name: str, description: str, instructions: str) -> Path:\n    path = root / "skills" / name / "SKILL.md"\n    path.parent.mkdir(parents=True)\n    path.write_text(\n        f"---\\nname: {name}\\ndescription: {description}\\n---\\n\\n{instructions}\\n",\n        encoding="utf-8",\n    )\n    return path\n\n\ndef test_resource_loader_resolves_skills_by_explicit_project_user_builtin_precedence(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    workspace.mkdir()\n    user_root = tmp_path / "user-resources"\n    _write_skill(user_root, "review", "user review", "USER")\n    _write_skill(workspace / ".omega", "review", "project review", "PROJECT")\n    trust = MemoryProjectTrust([workspace])\n    explicit = Resource.skill(\n        "review",\n        "explicit review",\n        "EXPLICIT",\n        source="command-line:review",\n    )\n    builtin = Resource.skill(\n        "review",\n        "built-in review",\n        "BUILTIN",\n        source="builtin:review",\n    )\n\n    loaded = ResourceLoader(\n        workspace,\n        user_root=user_root,\n        trust=trust,\n        explicit=[explicit],\n        builtins=[builtin],\n    ).load()\n\n    assert loaded.skills["review"].scope is ResourceScope.EXPLICIT\n    assert loaded.skills["review"].description == "explicit review"\n    assert loaded.skills["review"].content == "EXPLICIT"\n    assert loaded.skills["review"].source == "command-line:review"\n    assert loaded.evidence["skill:review"].sha256.startswith("sha256:")\n\n\ndef test_untrusted_project_resources_are_skipped_but_context_files_remain_independent(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    working_directory = workspace / "packages" / "app"\n    working_directory.mkdir(parents=True)\n    (workspace / "AGENTS.md").write_text("ROOT CONTEXT", encoding="utf-8")\n    (workspace / "packages" / "AGENTS.md").write_text(\n        "PACKAGE CONTEXT", encoding="utf-8"\n    )\n    _write_skill(workspace / ".omega", "deploy", "deploy safely", "PROTECTED")\n    (workspace / ".omega" / "APPEND_SYSTEM.md").write_text(\n        "PROTECTED APPEND", encoding="utf-8"\n    )\n    (workspace / ".omega" / "extensions").mkdir()\n    (workspace / ".omega" / "extensions" / "audit.py").write_text(\n        "PROTECTED_EXTENSION = True\\n", encoding="utf-8"\n    )\n    (workspace / ".omega" / "settings.json").write_text(\n        \'{"temperature": 0}\', encoding="utf-8"\n    )\n\n    loaded = ResourceLoader(\n        workspace,\n        working_directory=working_directory,\n        trust=MemoryProjectTrust(),\n    ).load()\n\n    assert [context.content for context in loaded.context_files] == [\n        "ROOT CONTEXT",\n        "PACKAGE CONTEXT",\n    ]\n    assert "deploy" not in loaded.skills\n    assert "audit" not in loaded.extensions\n    assert loaded.settings is None\n    assert loaded.append_system is None\n    assert any(path.endswith("SKILL.md") for path in loaded.skipped_protected)\n    assert any(path.endswith("APPEND_SYSTEM.md") for path in loaded.skipped_protected)\n    assert any(path.endswith("audit.py") for path in loaded.skipped_protected)\n    assert any(path.endswith("settings.json") for path in loaded.skipped_protected)\n\n    without_context = ResourceLoader(\n        workspace,\n        working_directory=working_directory,\n        trust=MemoryProjectTrust(),\n        context_enabled=False,\n    ).load()\n    assert without_context.context_files == ()\n\n\ndef test_prompt_assembly_is_ordered_and_discloses_only_active_skill_instructions(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    workspace.mkdir()\n    (workspace / "AGENTS.md").write_text("CONTEXT BODY", encoding="utf-8")\n    omega = workspace / ".omega"\n    omega.mkdir()\n    (omega / "APPEND_SYSTEM.md").write_text("APPENDED BODY", encoding="utf-8")\n    _write_skill(omega, "review", "review changes", "ACTIVE BODY")\n    _write_skill(omega, "deploy", "deploy changes", "INACTIVE BODY")\n    resources = ResourceLoader(\n        workspace,\n        trust=MemoryProjectTrust([workspace]),\n    ).load()\n\n    async def read_file(arguments: dict[str, object]) -> ToolResult:\n        return ToolResult("unused")\n\n    tool = Tool(\n        "read",\n        "Read a workspace text file",\n        {"type": "object", "properties": {"path": {"type": "string"}}},\n        read_file,\n    )\n    assembled = PromptAssembler("BUILT-IN BODY").assemble(\n        tools=[tool], resources=resources, active_skills=["review"]\n    )\n\n    positions = [\n        assembled.text.index(marker)\n        for marker in (\n            "BUILT-IN BODY",\n            "## Active Tools",\n            "## Available Skills",\n            "## Active Skill: review",\n            "## Context File:",\n            "## Appended System Instructions",\n        )\n    ]\n    assert positions == sorted(positions)\n    assert "review changes" in assembled.text\n    assert "deploy changes" in assembled.text\n    assert "ACTIVE BODY" in assembled.text\n    assert "INACTIVE BODY" not in assembled.text\n    assert assembled.prompt_hashes["effective_system"].startswith("sha256:")\n    assert set(assembled.resource_hashes) == set(resources.evidence)\n\n\ndef test_trusted_system_replacement_replaces_the_entire_default_assembly(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    omega = workspace / ".omega"\n    omega.mkdir(parents=True)\n    (workspace / "AGENTS.md").write_text("CONTEXT", encoding="utf-8")\n    (omega / "SYSTEM.md").write_text("REPLACEMENT BODY", encoding="utf-8")\n    (omega / "APPEND_SYSTEM.md").write_text("MUST NOT APPEND", encoding="utf-8")\n    _write_skill(omega, "review", "review changes", "SKILL BODY")\n    resources = ResourceLoader(\n        workspace,\n        trust=MemoryProjectTrust([workspace]),\n    ).load()\n\n    assembled = PromptAssembler("BUILT-IN BODY").assemble(\n        tools=[], resources=resources\n    )\n\n    assert assembled.text == "REPLACEMENT BODY\\n"\n    assert set(assembled.resource_hashes) == {"system:replacement"}\n\n\ndef test_loader_discovers_only_local_prompt_and_extension_sources_with_precedence(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    user_root = tmp_path / "user-resources"\n    for root, prompt, extension in (\n        (user_root, "USER {target}", "USER_EXTENSION = True\\n"),\n        (workspace / ".omega", "PROJECT {target}", "PROJECT_EXTENSION = True\\n"),\n    ):\n        (root / "prompts").mkdir(parents=True)\n        (root / "prompts" / "review.md").write_text(prompt, encoding="utf-8")\n        (root / "extensions").mkdir()\n        (root / "extensions" / "audit.py").write_text(extension, encoding="utf-8")\n    (workspace / ".omega" / "settings.json").write_text(\n        \'{"temperature": 0}\', encoding="utf-8"\n    )\n\n    loaded = ResourceLoader(\n        workspace,\n        user_root=user_root,\n        trust=MemoryProjectTrust([workspace]),\n    ).load()\n\n    assert loaded.prompt_templates["review"].scope is ResourceScope.PROJECT\n    assert loaded.expand_prompt("review", {"target": "app.py"}) == "PROJECT app.py"\n    assert loaded.extensions["audit"].scope is ResourceScope.PROJECT\n    assert loaded.settings is not None\n    assert set(loaded.evidence) >= {\n        "prompt:review",\n        "extension:audit",\n        "settings:project",\n    }\n\n\ndef test_project_trust_is_persisted_against_the_canonical_workspace_path(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    workspace.mkdir()\n    trust_file = tmp_path / "omega-data" / "trusted-projects.json"\n    store = JSONProjectTrustStore(trust_file)\n\n    store.approve(workspace / ".." / "project")\n\n    reloaded = JSONProjectTrustStore(trust_file)\n    assert reloaded.is_trusted(workspace.resolve()) is True\n    assert str(workspace.resolve()) in trust_file.read_text(encoding="utf-8")\n\n\ndef test_related_skill_resources_enter_context_only_after_activation(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    skill = _write_skill(\n        workspace / ".omega", "review", "review changes", "Read the checklist."\n    )\n    reference = skill.parent / "references" / "checklist.md"\n    reference.parent.mkdir()\n    reference.write_text("CHECK GENERATED FILES", encoding="utf-8")\n    resources = ResourceLoader(\n        workspace,\n        trust=MemoryProjectTrust([workspace]),\n    ).load()\n    assembler = PromptAssembler("BUILT IN")\n\n    inactive = assembler.assemble(tools=[], resources=resources)\n    active = assembler.assemble(\n        tools=[], resources=resources, active_skills=["review"]\n    )\n\n    assert "CHECK GENERATED FILES" not in inactive.text\n    assert "CHECK GENERATED FILES" in active.text\n    assert "references/checklist.md" in active.text\n\n\ndef test_local_resource_discovery_rejects_scoped_symlink_escape(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "project"\n    prompts = workspace / ".omega" / "prompts"\n    prompts.mkdir(parents=True)\n    outside = tmp_path / "outside.md"\n    outside.write_text("OUTSIDE", encoding="utf-8")\n    try:\n        (prompts / "escaped.md").symlink_to(outside)\n    except OSError as error:\n        pytest.skip(f"file symlinks unavailable: {error}")\n\n    with pytest.raises(ValueError, match="resource escapes its scope"):\n        ResourceLoader(\n            workspace,\n            trust=MemoryProjectTrust([workspace]),\n        ).load()\n'


In [ ]:
TEST_CODING_TOOLS_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nimport os\nimport stat\nfrom pathlib import Path\n\nimport pytest\n\nfrom agent_harness import (\n    BashOperations,\n    CompleteOutputKind,\n    FileArtifactStore,\n    ToolErrorCode,\n    create_coding_tool_preset,\n)\n\n\ndef test_file_tools_reject_parent_traversal_outside_the_workspace(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    preset = create_coding_tool_preset(workspace)\n\n    result = asyncio.run(\n        preset.tool("write").execute(\n            {"path": "../escaped.txt", "content": "must not escape"}\n        )\n    )\n\n    assert result.is_error is True\n    assert result.error_code is ToolErrorCode.EXECUTION_FAILED\n    assert "Workspace Boundary" in result.content\n    assert not (tmp_path / "escaped.txt").exists()\n\n\ndef test_file_tools_reject_symlink_escape_and_unsafe_nonexistent_parent(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    outside = tmp_path / "outside"\n    workspace.mkdir()\n    outside.mkdir()\n    (outside / "secret.txt").write_text("secret", encoding="utf-8")\n    try:\n        (workspace / "linked").symlink_to(outside, target_is_directory=True)\n    except OSError as error:\n        pytest.skip(f"directory symlinks unavailable: {error}")\n    preset = create_coding_tool_preset(workspace)\n\n    read_result = asyncio.run(\n        preset.tool("read").execute({"path": "linked/secret.txt"})\n    )\n    write_result = asyncio.run(\n        preset.tool("write").execute(\n            {"path": "linked/new/deep.txt", "content": "escape"}\n        )\n    )\n\n    assert read_result.error_code is ToolErrorCode.EXECUTION_FAILED\n    assert write_result.error_code is ToolErrorCode.EXECUTION_FAILED\n    assert not (outside / "new" / "deep.txt").exists()\n\n\ndef test_write_and_edit_reject_binary_mutation(tmp_path: Path) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    binary = workspace / "image.bin"\n    original = b"\\x00\\xff\\x10binary"\n    binary.write_bytes(original)\n    preset = create_coding_tool_preset(workspace)\n\n    write_result = asyncio.run(\n        preset.tool("write").execute({"path": "image.bin", "content": "text"})\n    )\n    edit_result = asyncio.run(\n        preset.tool("edit").execute(\n            {"path": "image.bin", "old_text": "binary", "new_text": "text"}\n        )\n    )\n\n    assert write_result.error_code is ToolErrorCode.EXECUTION_FAILED\n    assert edit_result.error_code is ToolErrorCode.EXECUTION_FAILED\n    assert binary.read_bytes() == original\n\n\ndef test_text_edit_commits_atomically_without_losing_file_permissions(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    source = workspace / "app.py"\n    source.write_text("VALUE = \'old\'\\n", encoding="utf-8")\n    source.chmod(0o640)\n    original_mode = stat.S_IMODE(source.stat().st_mode)\n    preset = create_coding_tool_preset(workspace)\n\n    result = asyncio.run(\n        preset.tool("edit").execute(\n            {"path": "app.py", "old_text": "old", "new_text": "new"}\n        )\n    )\n\n    assert result.is_error is False\n    assert source.read_text(encoding="utf-8") == "VALUE = \'new\'\\n"\n    assert stat.S_IMODE(source.stat().st_mode) == original_mode\n    assert list(workspace.glob(".*.omega-tmp")) == []\n\n\ndef test_bash_uses_fixed_workspace_and_filters_sensitive_environment(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    operations = BashOperations(\n        workspace,\n        environment={\n            "PATH": os.environ["PATH"],\n            "SAFE_VALUE": "visible",\n            "API_KEY": "hidden",\n            "AWS_ACCESS_KEY_ID": "also-hidden",\n            "ALLOWED_TOKEN": "deliberate",\n        },\n        allow_sensitive=["ALLOWED_TOKEN"],\n    )\n\n    result = asyncio.run(\n        operations.run(\n            "printf \'%s|%s|%s|%s\\\\n\' \\"$SAFE_VALUE\\" "\n            "\\"${API_KEY-unset}\\" \\"${AWS_ACCESS_KEY_ID-unset}\\" "\n            "\\"$ALLOWED_TOKEN\\"; pwd"\n        )\n    )\n\n    lines = result.stdout.splitlines()\n    assert lines[0] == "visible|unset|unset|deliberate"\n    assert Path(lines[1]).resolve() == workspace.resolve()\n    assert result.returncode == 0\n\n\ndef test_bash_timeout_and_cancellation_settle_the_host_process_promptly(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    operations = BashOperations(workspace)\n    preset = create_coding_tool_preset(workspace, bash_operations=operations)\n\n    timed_out = asyncio.run(\n        preset.tool("bash").execute(\n            {"command": "sleep 30", "timeout_seconds": 0.05}\n        )\n    )\n\n    async def cancel_running_command() -> None:\n        task = asyncio.create_task(operations.run("sleep 30"))\n        await asyncio.sleep(0.05)\n        task.cancel()\n        with pytest.raises(asyncio.CancelledError):\n            await asyncio.wait_for(task, timeout=2)\n\n    asyncio.run(cancel_running_command())\n    assert timed_out.error_code is ToolErrorCode.EXECUTION_FAILED\n    assert "exceeded" in timed_out.content\n\n\ndef test_large_bash_output_can_be_retained_as_a_sanitized_content_addressed_artifact(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    artifacts = FileArtifactStore(\n        tmp_path / "artifacts",\n        redact=lambda text: text.replace("secret", "[REDACTED]"),\n    )\n    preset = create_coding_tool_preset(\n        workspace,\n        artifact_store=artifacts,\n        artifact_threshold_bytes=8,\n    )\n\n    result = asyncio.run(\n        preset.tool("bash").execute({"command": "printf \'secret-value\'"})\n    )\n\n    assert result.complete_output is not None\n    assert result.complete_output.kind is CompleteOutputKind.ARTIFACT\n    reference = result.complete_output.reference\n    assert reference is not None and reference.startswith("sha256:")\n    assert artifacts.read_text(reference) == "[REDACTED]-value"\n\n\ndef test_replaceable_bash_backend_must_keep_the_preset_fixed_workspace(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    other = tmp_path / "other"\n    workspace.mkdir()\n    other.mkdir()\n\n    with pytest.raises(ValueError, match="BashOperations.*workspace"):\n        create_coding_tool_preset(\n            workspace,\n            bash_operations=BashOperations(other),\n        )\n'


In [ ]:
TEST_CODING_AGENT_SOURCE = 'from __future__ import annotations\n\nimport asyncio\nfrom pathlib import Path\n\nimport pytest\n\nfrom agent_harness import (\n    AgentRuntime,\n    MemoryProjectTrust,\n    ModelEnd,\n    ModelSpec,\n    ResourceLoader,\n    ScriptedModelAdapter,\n    StopReason,\n    TextDelta,\n    ToolCallDelta,\n    create_coding_agent,\n)\n\n\ndef test_coding_agent_uses_one_session_to_read_edit_and_verify_a_real_workspace(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    (workspace / "app.py").write_text("VALUE = 1\\n", encoding="utf-8")\n    (workspace / "AGENTS.md").write_text(\n        "Keep the verification local and deterministic.", encoding="utf-8"\n    )\n    adapter = ScriptedModelAdapter(\n        [\n            [\n                ToolCallDelta(0, "read-1", "read", \'{"path":"app.py"}\'),\n                ModelEnd(StopReason.TOOL_USE),\n            ],\n            [\n                ToolCallDelta(\n                    0,\n                    "edit-1",\n                    "edit",\n                    \'{"path":"app.py","old_text":"VALUE = 1",\'\n                    \'"new_text":"VALUE = 2"}\',\n                ),\n                ModelEnd(StopReason.TOOL_USE),\n            ],\n            [\n                ToolCallDelta(\n                    0,\n                    "bash-1",\n                    "bash",\n                    \'{"command":"grep -q \\\'VALUE = 2\\\' app.py"}\',\n                ),\n                ModelEnd(StopReason.TOOL_USE),\n            ],\n            [TextDelta("Updated and verified app.py"), ModelEnd(StopReason.COMPLETE)],\n        ]\n    )\n    resources = ResourceLoader(\n        workspace,\n        trust=MemoryProjectTrust([workspace]),\n    )\n    agent = create_coding_agent(\n        adapter,\n        ModelSpec("scripted/coding-agent"),\n        workspace,\n        resource_loader=resources,\n    )\n\n    result = asyncio.run(agent.session.run("Update VALUE to 2 and verify it"))\n\n    assert (workspace / "app.py").read_text(encoding="utf-8") == "VALUE = 2\\n"\n    assert result.outcome.message.content[0].text == "Updated and verified app.py"\n    assert [tool.name for tool in agent.preset.tools] == [\n        "read",\n        "write",\n        "edit",\n        "bash",\n    ]\n    assert adapter.received_requests[0].messages[0].content[0].text.startswith(\n        "You are Omega"\n    )\n    assert dict(agent.startup_evidence)["context:0:" + str(\n        (workspace / "AGENTS.md").resolve()\n    )].startswith("sha256:")\n\n\ndef test_general_runtime_still_installs_no_tools() -> None:\n    runtime = AgentRuntime(\n        ScriptedModelAdapter([TextDelta("done"), ModelEnd(StopReason.COMPLETE)]),\n        ModelSpec("scripted/general"),\n    )\n\n    assert runtime.tools == ()\n\n\ndef test_coding_agent_rejects_resource_loader_from_another_workspace(\n    tmp_path: Path,\n) -> None:\n    workspace = tmp_path / "workspace"\n    other = tmp_path / "other"\n    workspace.mkdir()\n    other.mkdir()\n    adapter = ScriptedModelAdapter(\n        [TextDelta("unused"), ModelEnd(StopReason.COMPLETE)]\n    )\n\n    with pytest.raises(ValueError, match="ResourceLoader.*workspace"):\n        create_coding_agent(\n            adapter,\n            ModelSpec("scripted/mismatched-resources"),\n            workspace,\n            resource_loader=ResourceLoader(other),\n        )\n'


In [ ]:
TEST_CONTRACT_SOURCE = 'from __future__ import annotations\n\nfrom pathlib import Path\n\nfrom agent_harness import (\n    AgentRuntime,\n    MemoryProjectTrust,\n    ModelEnd,\n    ModelSpec,\n    ResourceLoader,\n    ScriptedModelAdapter,\n    StopReason,\n    TextDelta,\n    create_coding_tool_preset,\n)\n\n\ndef test_chapter_08_public_contract_and_optional_tool_preset(tmp_path: Path) -> None:\n    workspace = tmp_path / "workspace"\n    workspace.mkdir()\n    resources = ResourceLoader(\n        workspace,\n        trust=MemoryProjectTrust(),\n    ).load()\n    preset = create_coding_tool_preset(workspace)\n    general = AgentRuntime(\n        ScriptedModelAdapter([TextDelta("done"), ModelEnd(StopReason.COMPLETE)]),\n        ModelSpec("scripted/ch08-contract"),\n    )\n\n    assert resources.skills == {}\n    assert general.tools == ()\n    assert [tool.name for tool in preset.tools] == ["read", "write", "edit", "bash"]\n\n\ndef test_checkpoint_documents_host_authority_without_a_sandbox_claim() -> None:\n    readme = (Path(__file__).parents[1] / "README.md").read_text(encoding="utf-8")\n\n    assert "real host process" in readme\n    assert "Neither Bash nor Extensions are a sandbox guarantee" in readme\n'


In [ ]:
omega = workspace / '.omega'
skill_file = omega / 'skills' / 'review' / 'SKILL.md'
skill_file.parent.mkdir(parents=True)
skill_file.write_text(
    '---\nname: review\ndescription: Review changes\n---\n\nCheck the diff.\n',
    encoding='utf-8',
)
trusted = chapter8.MemoryProjectTrust([workspace])
loaded = chapter8.ResourceLoader(workspace, trust=trusted).load()
assembled = chapter8.PromptAssembler('BUILT IN').assemble(
    tools=preset.tools, resources=loaded, active_skills=['review']
)
assert assembled.text.index('BUILT IN') < assembled.text.index('## Active Tools')
assert 'Check the diff.' in assembled.text
assert assembled.resource_hashes['skill:review'].startswith('sha256:')


## Observable Trace

Resolved resource hashes enter the existing frozen Run Snapshot. This makes startup provenance available to the later Run Trace without coupling discovery to Runtime.

In [ ]:
trace_adapter = chapter8.ScriptedModelAdapter([
    chapter8.TextDelta('traceable coding session'),
    chapter8.ModelEnd(chapter8.StopReason.COMPLETE),
])
coding_agent = chapter8.create_coding_agent(
    trace_adapter,
    chapter8.ModelSpec('scripted/ch08-trace'),
    workspace,
    resource_loader=chapter8.ResourceLoader(workspace, trust=trusted),
)
trace_handle = coding_agent.session.start('inspect the workspace')
trace_result = await trace_handle.result()
assert trace_result.outcome.stop_reason is chapter8.StopReason.COMPLETE
assert trace_handle.snapshot.resource_hashes
dict(coding_agent.startup_evidence)


## Failure Boundaries and Trade-offs

Unapproved protected project inputs are skipped in non-interactive construction, while Context Files can be disabled separately. Canonical path validation follows symlinks and nonexistent parents before file creation. Binary mutation fails clearly. Bash resolution, timeout, cancellation, output direction, and sensitive-environment filtering are explicit, but a host Bash process can still access everything available to its process authority.

In [ ]:
untrusted = chapter8.ResourceLoader(
    workspace, trust=chapter8.MemoryProjectTrust()
).load()
assert 'review' not in untrusted.skills
assert untrusted.context_files
escape = await preset.tool('write').execute({
    'path': '../escape.txt', 'content': 'blocked'
})
assert escape.error_code is chapter8.ToolErrorCode.EXECUTION_FAILED
assert 'Workspace Boundary' in escape.content


## Checkpoint Export and Verification

The Notebook now removes its temporary import, exports only its tagged cells over Chapter 7, and runs compilation, offline installation, import, and every cumulative regression test before publishing Chapter 8.

In [ ]:
sys.path.remove(temporary_package.name)
temporary_package.cleanup()
temporary_workspace.cleanup()
for module_name in tuple(sys.modules):
    if module_name == 'agent_harness' or module_name.startswith('agent_harness.'):
        del sys.modules[module_name]


In [ ]:
from course.tools.checkpoint import checkpoint_drift, export_checkpoint

checkpoint_result = export_checkpoint(
    ROOT / 'course' / 'notebooks' / '08_coding_agent.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch08',
)
assert checkpoint_result.gates == ('compile', 'install', 'import', 'tests')
assert checkpoint_drift(
    ROOT / 'course' / 'notebooks' / '08_coding_agent.ipynb',
    ROOT / 'course' / 'checkpoints' / 'ch08',
) == ()
checkpoint_result


## Public API Summary

Chapter 8 adds `Resource`, `ResourceLoader`, Project Trust adapters, `PromptAssembler`, `CodingToolPreset`, `BashOperations`, `FileArtifactStore`, and `create_coding_agent`. General Runtime construction still installs no Tools; coding construction explicitly installs the preset and feeds prompt/resource hashes into the same `AgentSession` contracts.

In [ ]:
public_chapter8 = (
    chapter8.ResourceLoader, chapter8.PromptAssembler,
    chapter8.CodingToolPreset, chapter8.BashOperations,
    chapter8.FileArtifactStore, chapter8.create_coding_agent,
) if 'chapter8' in globals() else (
    'ResourceLoader', 'PromptAssembler', 'CodingToolPreset',
    'BashOperations', 'FileArtifactStore', 'create_coding_agent',
)
public_chapter8
